# Phase 4 — Model Evaluation & Explainability
### Hospital Operations & Revenue Risk Intelligence Platform

This notebook evaluates the machine learning models developed in **Phase 3**.

Two models are evaluated:
1. **Visit Risk Classification Model** — Predicts whether a hospital visit is Low, Medium, or High risk.
2. **Claim Outcome Model** — Predicts whether an insurance claim will be Paid, Pending, or Rejected.

Evaluation includes:
- Data preparation
- Ensuring feature schema consistency with trained models
- Confusion matrices
- Classification reports (Precision, Recall, F1-score)
- Business metrics required by the capstone

Since the models were trained using **scikit‑learn Pipelines**, we pass the raw feature columns directly to the models without manual encoding.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import joblib
from sklearn.metrics import confusion_matrix, classification_report

## 2. Load Modeling Dataset
The dataset `model_table.csv` was generated during **Phase 2 feature engineering**.

In [2]:
df = pd.read_csv(r'.\data\model_table.csv')
print('Dataset Shape:', df.shape)
df.head()

Dataset Shape: (25000, 33)


,visit_id,patient_id,visit_date,department,visit_type,length_of_stay_hours,risk_score,doctor_id,age,gender,...,approval_ratio,payment_delay_flag,visit_frequency,avg_los_per_patient,is_rejected,provider_rejection_rate,age_group,los_category,visit_intensity,dept_high_risk_rate
0,1,756,2025-10-18,Cardiology,ER,3.48,Low,169,90,M,...,0.0,0,2,3.725000,1,0.148655,Senior,Short,0.030769,0.189950
1,2,4102,2025-04-06,Orthopedics,OPD,15.31,High,148,30,M,...,1.0,0,4,32.025000,0,0.156915,Adult,Medium,-0.019417,0.202209
2,3,2964,2025-07-13,ICU,ER,34.36,Low,153,25,F,...,1.0,0,4,20.542500,0,0.149678,Adult,Long,0.444444,0.207923
3,4,4496,2025-11-19,Cardiology,ER,37.89,High,119,75,M,...,1.0,0,7,28.165714,0,0.152480,Senior,Long,-0.112903,0.189950
4,5,1930,2025-03-29,General,ICU,16.78,Medium,118,80,M,...,1.0,0,5,22.988000,0,0.149678,Senior,Medium,5.000000,0.198439


## Part A — Visit Risk Model Evaluation

### Load Trained Risk Model
The model was saved during Phase 3 using `joblib`.

In [4]:
risk_model = joblib.load(r'.\data\risk_model.pkl')

### Identify Required Feature Columns
The pipeline stores the feature schema used during training.
We extract those columns to ensure consistency during evaluation.

In [5]:
features = risk_model.feature_names_in_
print('Model expects the following features:')
print(features)

Model expects the following features:
['age' 'gender' 'department' 'visit_type' 'length_of_stay_hours'
 'visit_frequency' 'avg_los_per_patient' 'provider_rejection_rate'
 'days_since_registration' 'visit_month' 'visit_dayofweek']


### Build Feature Matrix
Only the required columns are passed to the pipeline model.

In [6]:
X = df[features]
y = df['risk_score']

### Time-Based Train/Test Split
Following the capstone requirement, the earliest **80% of records are used for training** and the latest **20% for testing**.

In [7]:
split_index = int(len(X)*0.8)

X_train_risk = X.iloc[:split_index]
X_test_risk  = X.iloc[split_index:]

y_train_risk = y.iloc[:split_index]
y_test_risk  = y.iloc[split_index:]

### Generate Predictions

In [8]:
train_pred_risk = risk_model.predict(X_train_risk)
test_pred_risk  = risk_model.predict(X_test_risk)

### Evaluate Risk Model

In [9]:
print('Train Confusion Matrix')
print(confusion_matrix(y_train_risk, train_pred_risk))

print('\nTest Confusion Matrix')
print(confusion_matrix(y_test_risk, test_pred_risk))

print('\nClassification Report')
print(classification_report(y_test_risk, test_pred_risk))

Train Confusion Matrix
[[3239  646  150]
 [  47 9673  278]
 [  45  944 4978]]

Test Confusion Matrix
[[ 795  160   44]
 [  15 2372   85]
 [  13  216 1300]]

Classification Report
              precision    recall  f1-score   support

        High       0.97      0.80      0.87       999
         Low       0.86      0.96      0.91      2472
      Medium       0.91      0.85      0.88      1529

    accuracy                           0.89      5000
   macro avg       0.91      0.87      0.89      5000
weighted avg       0.90      0.89      0.89      5000



### Business Metric — High Risk Recall
Hospitals prioritize correctly identifying **High Risk visits** to allocate resources effectively.

In [10]:
report = classification_report(y_test_risk, test_pred_risk, output_dict=True)

if 'High' in report:
    print('High Risk Recall:', report['High']['recall'])

High Risk Recall: 0.7957957957957958


## Part B — Claim Outcome Model Evaluation

### Load Claim Model

In [11]:
claim_model = joblib.load(r'.\data\claim_model.pkl')

### Prepare Claim Model Features

In [12]:
claim_features = claim_model.feature_names_in_

X_claim = df[claim_features].copy()
y_claim = df['claim_status']

# Encode categorical columns to numeric
if 'gender' in X_claim.columns:
    X_claim['gender'] = X_claim['gender'].map({'M': 1, 'F': 0}).fillna(0)

if 'department' in X_claim.columns:
    X_claim['department'] = pd.factorize(X_claim['department'])[0]

if 'visit_type' in X_claim.columns:
    X_claim['visit_type'] = pd.factorize(X_claim['visit_type'])[0]

### Time-Based Split for Claim Model

In [13]:
X_train_claim = X_claim.iloc[:split_index]
X_test_claim  = X_claim.iloc[split_index:]

y_train_claim = y_claim.iloc[:split_index]
y_test_claim  = y_claim.iloc[split_index:]

### Generate Claim Predictions

In [14]:
train_pred_claim = claim_model.predict(X_train_claim)
test_pred_claim  = claim_model.predict(X_test_claim)

### Evaluate Claim Model

In [15]:
print('Train Confusion Matrix')
print(confusion_matrix(y_train_claim, train_pred_claim))

print('\nTest Confusion Matrix')
print(confusion_matrix(y_test_claim, test_pred_claim))

print('\nClassification Report')
print(classification_report(y_test_claim, test_pred_claim))

Train Confusion Matrix
[[10927    82   912]
 [ 2040  2531   463]
 [  992    37  2016]]

Test Confusion Matrix
[[2762   18  239]
 [ 474  651  104]
 [ 264    9  479]]

Classification Report
              precision    recall  f1-score   support

        Paid       0.79      0.91      0.85      3019
     Pending       0.96      0.53      0.68      1229
    Rejected       0.58      0.64      0.61       752

    accuracy                           0.78      5000
   macro avg       0.78      0.69      0.71      5000
weighted avg       0.80      0.78      0.77      5000



### Business Metric — Rejected Claim Recall
Detecting **Rejected claims early** helps hospitals reduce revenue leakage.

In [16]:
report_claim = classification_report(y_test_claim, test_pred_claim, output_dict=True)

if 'Rejected' in report_claim:
    print('Rejected Claim Recall:', report_claim['Rejected']['recall'])

Rejected Claim Recall: 0.636968085106383


## 3. Resampling: Handle Class Imbalance with SMOTE or Undersampling

Class imbalance can severely impact model performance, especially for minority classes.
We address this using **SMOTE (Synthetic Minority Oversampling Technique)** or **undersampling**.

- **SMOTE**: Generates synthetic examples of minority classes to balance the dataset
- **Undersampling**: Reduces majority class samples to balance with minority classes

In [18]:
%pip install -q imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import json

# Encode categorical features in Risk Model data for SMOTE
X_train_risk_enc = X_train_risk.copy()
X_test_risk_enc = X_test_risk.copy()

if 'gender' in X_train_risk_enc.columns:
    X_train_risk_enc['gender'] = X_train_risk_enc['gender'].map({'M': 1, 'F': 0}).fillna(0)
    X_test_risk_enc['gender'] = X_test_risk_enc['gender'].map({'M': 1, 'F': 0}).fillna(0)

if 'department' in X_train_risk_enc.columns:
    dept_mapping = {val: idx for idx, val in enumerate(pd.concat([X_train_risk_enc['department'], X_test_risk_enc['department']]).unique())}
    X_train_risk_enc['department'] = X_train_risk_enc['department'].map(dept_mapping).fillna(0)
    X_test_risk_enc['department'] = X_test_risk_enc['department'].map(dept_mapping).fillna(0)

if 'visit_type' in X_train_risk_enc.columns:
    visit_mapping = {val: idx for idx, val in enumerate(pd.concat([X_train_risk_enc['visit_type'], X_test_risk_enc['visit_type']]).unique())}
    X_train_risk_enc['visit_type'] = X_train_risk_enc['visit_type'].map(visit_mapping).fillna(0)
    X_test_risk_enc['visit_type'] = X_test_risk_enc['visit_type'].map(visit_mapping).fillna(0)

# Convert all columns to numeric, handling any remaining object columns
for col in X_train_risk_enc.columns:
    X_train_risk_enc[col] = pd.to_numeric(X_train_risk_enc[col], errors='coerce').fillna(0)
    X_test_risk_enc[col] = pd.to_numeric(X_test_risk_enc[col], errors='coerce').fillna(0)

# Load claim features from phase3/claim_features.json
with open(r'F:\AI ML\capstone\phase3\claim_features.json', 'r') as f:
    claim_json = json.load(f)
claim_features_json = list(claim_json['features'].keys())

# Only use features that exist in the current DataFrame
claim_features_json_existing = [col for col in claim_features_json if col in X_train_claim.columns]
missing_features = [col for col in claim_features_json if col not in X_train_claim.columns]
if missing_features:
    print(f'Warning: The following features from claim_features.json are missing in the data and will be skipped: {missing_features}')

X_train_claim_enc = X_train_claim[claim_features_json_existing].copy()
X_test_claim_enc = X_test_claim[claim_features_json_existing].copy()

for col in X_train_claim_enc.columns:
    X_train_claim_enc[col] = pd.to_numeric(X_train_claim_enc[col], errors='coerce').fillna(0)
    X_test_claim_enc[col] = pd.to_numeric(X_test_claim_enc[col], errors='coerce').fillna(0)

print("=" * 60)
print("RISK MODEL - Class Distribution Before Resampling")
print("=" * 60)
print(y_train_risk.value_counts())

# Apply SMOTE to Risk Model
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_risk_smote, y_train_risk_smote = smote.fit_resample(X_train_risk_enc, y_train_risk)

print("\nRISK MODEL - Class Distribution After SMOTE")
print("=" * 60)
print(pd.Series(y_train_risk_smote).value_counts())

# Train model with SMOTE data
risk_model_smote = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
risk_model_smote.fit(X_train_risk_smote, y_train_risk_smote)
y_pred_risk_smote = risk_model_smote.predict(X_test_risk_enc)

print("\n" + "=" * 60)
print("CLAIM MODEL - Class Distribution Before Resampling")
print("=" * 60)
print(y_train_claim.value_counts())

# Apply SMOTE to Claim Model (with JSON features)
X_train_claim_smote, y_train_claim_smote = smote.fit_resample(X_train_claim_enc, y_train_claim)

print("\nCLAIM MODEL - Class Distribution After SMOTE")
print("=" * 60)
print(pd.Series(y_train_claim_smote).value_counts())

# Train model with SMOTE data (with JSON features)
claim_model_smote = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
claim_model_smote.fit(X_train_claim_smote, y_train_claim_smote)
y_pred_claim_smote = claim_model_smote.predict(X_test_claim_enc)

print("\n✓ SMOTE resampling completed for both models")

RISK MODEL - Class Distribution Before Resampling
risk_score
Low       9998
Medium    5967
High      4035
Name: count, dtype: int64

RISK MODEL - Class Distribution After SMOTE
risk_score
Low       9998
High      9998
Medium    9998
Name: count, dtype: int64

CLAIM MODEL - Class Distribution Before Resampling
claim_status
Paid        11921
Pending      5034
Rejected     3045
Name: count, dtype: int64

CLAIM MODEL - Class Distribution After SMOTE
claim_status
Rejected    11921
Paid        11921
Pending     11921
Name: count, dtype: int64

✓ SMOTE resampling completed for both models


## 4. Evaluate SMOTE Models

Compare the SMOTE-trained models against the original models to assess performance improvement.

In [20]:
# Risk Model: Original vs SMOTE

print("=" * 70)
print("RISK MODEL — ORIGINAL MODEL PERFORMANCE")
print("=" * 70)
print(classification_report(y_test_risk, test_pred_risk))

print("\n" + "=" * 70)
print("RISK MODEL — SMOTE MODEL PERFORMANCE")
print("=" * 70)
print(classification_report(y_test_risk, y_pred_risk_smote))

# Compare High Risk Recall
report_original = classification_report(y_test_risk, test_pred_risk, output_dict=True)
report_smote = classification_report(y_test_risk, y_pred_risk_smote, output_dict=True)

if 'High' in report_original and 'High' in report_smote:
    print("\n" + "=" * 70)
    print("HIGH RISK RECALL COMPARISON")
    print("=" * 70)
    print(f"Original Model Recall: {report_original['High']['recall']:.4f}")
    print(f"SMOTE Model Recall:    {report_smote['High']['recall']:.4f}")
    improvement = (report_smote['High']['recall'] - report_original['High']['recall']) * 100
    print(f"Improvement: {improvement:+.2f}%")

RISK MODEL — ORIGINAL MODEL PERFORMANCE
              precision    recall  f1-score   support

        High       0.97      0.80      0.87       999
         Low       0.86      0.96      0.91      2472
      Medium       0.91      0.85      0.88      1529

    accuracy                           0.89      5000
   macro avg       0.91      0.87      0.89      5000
weighted avg       0.90      0.89      0.89      5000


RISK MODEL — SMOTE MODEL PERFORMANCE
              precision    recall  f1-score   support

        High       0.20      0.09      0.13       999
         Low       0.51      0.74      0.60      2472
      Medium       0.34      0.21      0.26      1529

    accuracy                           0.45      5000
   macro avg       0.35      0.35      0.33      5000
weighted avg       0.39      0.45      0.40      5000


HIGH RISK RECALL COMPARISON
Original Model Recall: 0.7958
SMOTE Model Recall:    0.0921
Improvement: -70.37%


In [21]:
# Claim Model: Original vs SMOTE

print("\n" + "=" * 70)
print("CLAIM MODEL — ORIGINAL MODEL PERFORMANCE")
print("=" * 70)
print(classification_report(y_test_claim, test_pred_claim))

print("\n" + "=" * 70)
print("CLAIM MODEL — SMOTE MODEL PERFORMANCE")
print("=" * 70)
print(classification_report(y_test_claim, y_pred_claim_smote))

# Compare Rejected Claim Recall
report_claim_original = classification_report(y_test_claim, test_pred_claim, output_dict=True)
report_claim_smote = classification_report(y_test_claim, y_pred_claim_smote, output_dict=True)

if 'Rejected' in report_claim_original and 'Rejected' in report_claim_smote:
    print("\n" + "=" * 70)
    print("REJECTED CLAIM RECALL COMPARISON")
    print("=" * 70)
    print(f"Original Model Recall: {report_claim_original['Rejected']['recall']:.4f}")
    print(f"SMOTE Model Recall:    {report_claim_smote['Rejected']['recall']:.4f}")
    improvement = (report_claim_smote['Rejected']['recall'] - report_claim_original['Rejected']['recall']) * 100
    print(f"Improvement: {improvement:+.2f}%")

print("\n" + "=" * 70)
print("✓ SMOTE Model Evaluation Complete")
print("=" * 70)


CLAIM MODEL — ORIGINAL MODEL PERFORMANCE
              precision    recall  f1-score   support

        Paid       0.79      0.91      0.85      3019
     Pending       0.96      0.53      0.68      1229
    Rejected       0.58      0.64      0.61       752

    accuracy                           0.78      5000
   macro avg       0.78      0.69      0.71      5000
weighted avg       0.80      0.78      0.77      5000


CLAIM MODEL — SMOTE MODEL PERFORMANCE
              precision    recall  f1-score   support

        Paid       0.63      0.78      0.70      3019
     Pending       0.27      0.12      0.17      1229
    Rejected       0.25      0.23      0.24       752

    accuracy                           0.54      5000
   macro avg       0.38      0.38      0.37      5000
weighted avg       0.48      0.54      0.50      5000


REJECTED CLAIM RECALL COMPARISON
Original Model Recall: 0.6370
SMOTE Model Recall:    0.2314
Improvement: -40.56%

✓ SMOTE Model Evaluation Complete


## 5. Model Selection & Final Recommendation

Based on SMOTE evaluation results, select the best model for production deployment.


In [ ]:
print("\n" + "=" * 80)
print("FINAL MODEL RECOMMENDATION")
print("=" * 80)

# Risk Model Analysis
print("\n[RISK MODEL ANALYSIS]")
print("-" * 80)
risk_original_recall = report_original['High']['recall'] if 'High' in report_original else 0
risk_smote_recall = report_smote['High']['recall'] if 'High' in report_smote else 0
risk_improvement = (risk_smote_recall - risk_original_recall) * 100

print(f"✓ Original Model — High Risk Recall: {risk_original_recall:.4f}")
print(f"✓ SMOTE Model — High Risk Recall:    {risk_smote_recall:.4f}")
print(f"→ Improvement: {risk_improvement:+.2f}%")

if risk_improvement > 0:
    print("✓ RECOMMENDATION: Use SMOTE Risk Model (Better at catching High Risk visits)")
else:
    print("✗ RECOMMENDATION: Use Original Risk Model")

# Claim Model Analysis
print("\n[CLAIM MODEL ANALYSIS]")
print("-" * 80)
claim_original_recall = report_claim_original['Rejected']['recall'] if 'Rejected' in report_claim_original else 0
claim_smote_recall = report_claim_smote['Rejected']['recall'] if 'Rejected' in report_claim_smote else 0
claim_improvement = (claim_smote_recall - claim_original_recall) * 100

print(f"✓ Original Model — Rejected Claim Recall: {claim_original_recall:.4f}")
print(f"✓ SMOTE Model — Rejected Claim Recall:    {claim_smote_recall:.4f}")
print(f"→ Improvement: {claim_improvement:+.2f}%")

if claim_improvement > 0:
    print("✓ RECOMMENDATION: Use SMOTE Claim Model (Better at catching Rejected claims)")
else:
    print("✗ RECOMMENDATION: Use Original Claim Model")

# Final Decision
print("\n" + "=" * 80)
print("DEPLOYMENT DECISION")
print("=" * 80)

if risk_improvement > 0 and claim_improvement > 0:
    print("✓ DEPLOY SMOTE MODELS FOR PRODUCTION")
    print("  • Risk Model: Detects more High Risk visits for resource allocation")
    print("  • Claim Model: Detects more Rejected claims to reduce revenue leakage")
    print("\nSaving SMOTE models...")
    
    joblib.dump(risk_model_smote, r'.\data\risk_model_smote.pkl', compress=3)
    print("✓ Risk SMOTE model saved: risk_model_smote.pkl")
    
    joblib.dump(claim_model_smote, r'.\data\claim_model_smote.pkl', compress=3)
    print("✓ Claim SMOTE model saved: claim_model_smote.pkl")
    
else:
    print("✓ MIXED RESULTS - Saving models based on individual improvements")
    print("\nModel Selection:")
    
    if risk_improvement > 0:
        print("  ✓ Risk: Use SMOTE Model")
        joblib.dump(risk_model_smote, r'.\data\risk_modelpkl', compress=3)
        print("    → Saving: risk_model.pkl")
    else:
        print("  ✗ Risk: Keep Original Model (risk_model.pkl)")
    
    if claim_improvement > 0:
        print("  ✓ Claim: Use SMOTE Model")
        joblib.dump(claim_model_smote, r'.\data\claim_model.pkl', compress=3)
        print("    → Saving: claim_model.pkl")
    else:
        print("  ✗ Claim: Keep Original Model (claim_model.pkl)")

print("\n" + "=" * 80)
print("✓ PHASE 4 EVALUATION COMPLETE")
print("=" * 80)



FINAL MODEL RECOMMENDATION

[RISK MODEL ANALYSIS]
--------------------------------------------------------------------------------
✓ Original Model — High Risk Recall: 0.7958
✓ SMOTE Model — High Risk Recall:    0.0921
→ Improvement: -70.37%
✗ RECOMMENDATION: Use Original Risk Model

[CLAIM MODEL ANALYSIS]
--------------------------------------------------------------------------------
✓ Original Model — Rejected Claim Recall: 0.6370
✓ SMOTE Model — Rejected Claim Recall:    0.2314
→ Improvement: -40.56%
✗ RECOMMENDATION: Use Original Claim Model

DEPLOYMENT DECISION
✓ MIXED RESULTS - Saving models based on individual improvements

Model Selection:
  ✗ Risk: Keep Original Model (risk_model.pkl)
  ✗ Claim: Keep Original Model (claim_model.pkl)

✓ PHASE 4 EVALUATION COMPLETE
